# 03 Cohort Construction

This notebook builds the adult ICU cohort and implements a reproducible Sepsis-3-aligned labeling pipeline using suspected infection and SOFA score increase. It saves intermediate artifacts so the cohort definition can be audited and reused by downstream notebooks.

## Important methodological note

The default implementation below is **Sepsis-3 aligned and leakage-safe**, but its exact SOFA variable extraction depends on local `D_ITEMS.csv` and `D_LABITEMS.csv` label matching. Before paper submission, the resolved item sets should be clinically validated against the local MIMIC dictionary tables and refined if needed.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy

In [ ]:
import pandas as pd

from src.utils.config import load_config
from src.utils.paths import ensure_directories, resolve_project_paths
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.data_processing.io import list_available_tables, validate_required_tables
from src.data_processing.cohort import build_base_icu_cohort, summarize_cohort, create_patient_level_splits
from src.data_processing.sepsis3 import (
    compute_sofa_hourly,
    derive_sepsis_labels,
    derive_suspected_infection,
    detect_antibiotic_orders,
    detect_culture_orders,
    extract_sofa_measurements,
)

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)

available_tables = list_available_tables(paths['extracted_data_dir'])
required_status = validate_required_tables(paths['extracted_data_dir'], config['dataset']['required_tables'])
assert all(required_status.values()), 'Some required tables are missing. Run 01_dataset_setup first.'

available_tables[:20]

## Build the base ICU cohort

In [ ]:
cohort_df = build_base_icu_cohort(
    extracted_dir=paths['extracted_data_dir'],
    adult_age_min=config['cohort']['adult_age_min'],
    min_icu_los_hours=config['cohort']['min_icu_los_hours'],
    first_icu_only=config['cohort']['first_icu_only'],
    low_memory=config['dataset']['low_memory'],
)
cohort_summary = summarize_cohort(cohort_df)
cohort_summary

In [ ]:
cohort_df.head()

## Detect suspected infection

This step implements the Sepsis-3 temporal pairing logic between antibiotic administration and culture sampling.

In [ ]:
antibiotics_df = detect_antibiotic_orders(
    extracted_dir=paths['extracted_data_dir'],
    antibiotic_keywords=config['sepsis3']['antibiotic_keywords'],
    low_memory=config['dataset']['low_memory'],
)
cultures_df = detect_culture_orders(
    extracted_dir=paths['extracted_data_dir'],
    low_memory=config['dataset']['low_memory'],
)
suspected_infection_df = derive_suspected_infection(
    antibiotics=antibiotics_df,
    cultures=cultures_df,
    culture_after_antibiotic_hours=config['sepsis3']['suspicion_window']['culture_after_antibiotic_hours'],
    antibiotic_after_culture_hours=config['sepsis3']['suspicion_window']['antibiotic_after_culture_hours'],
)
suspected_infection_df.head()

## Resolve SOFA variables and compute hourly proxy SOFA

SOFA variables are resolved from dictionary tables when available. The resulting hourly table is the basis for sepsis onset labeling and is saved for auditability.

In [ ]:
measurements = extract_sofa_measurements(
    extracted_dir=paths['extracted_data_dir'],
    cohort=cohort_df,
    lab_item_keywords=config['sepsis3']['lab_item_keywords'],
    chart_item_keywords=config['sepsis3']['chart_item_keywords'],
    vasopressor_keywords=config['sepsis3']['vasopressor_keywords'],
    chunksize=config['dataset']['chunksize'],
    low_memory=config['dataset']['low_memory'],
)
sofa_hourly_df = compute_sofa_hourly(measurements, cohort_df)
sofa_hourly_df.head()

In [ ]:
measurements['lab_itemids'].head(), measurements['chart_itemids'].head()

## Derive sepsis onset labels

In [ ]:
labels_df = derive_sepsis_labels(
    cohort=cohort_df,
    suspected_infection=suspected_infection_df,
    sofa_hourly=sofa_hourly_df,
    baseline_window_hours=config['sepsis3']['baseline_window_hours'],
    sofa_delta_threshold=config['sepsis3']['sofa_delta_threshold'],
    sofa_before_suspicion_hours=config['sepsis3']['onset_window']['sofa_before_suspicion_hours'],
    sofa_after_suspicion_hours=config['sepsis3']['onset_window']['sofa_after_suspicion_hours'],
)
labels_df.head()

In [ ]:
label_summary = {
    'cohort_stays': int(cohort_df['ICUSTAY_ID'].nunique()),
    'suspected_infection_admissions': int(suspected_infection_df['HADM_ID'].nunique()) if not suspected_infection_df.empty else 0,
    'sepsis_positive_stays': int(labels_df.loc[labels_df['sepsis3_label'] == 1, 'ICUSTAY_ID'].nunique()),
    'positive_rate': float(labels_df['sepsis3_label'].mean()) if not labels_df.empty else 0.0,
}
label_summary

## Create patient-level splits

In [ ]:
splits_df = create_patient_level_splits(
    cohort=cohort_df,
    val_size=config['training']['val_size'],
    test_size=config['training']['test_size'],
    random_state=config['cohort']['patient_split_random_state'],
)
splits_df['split'].value_counts()

## Save reproducible artifacts

In [ ]:
output_dir = paths['processed_data_dir'] / '03_cohort_construction'
artifact_bundle = {
    'cohort': cohort_df,
    'antibiotics': antibiotics_df,
    'cultures': cultures_df,
    'suspected_infection': suspected_infection_df,
    'sofa_hourly': sofa_hourly_df,
    'sepsis_labels': labels_df,
    'patient_splits': splits_df,
    'resolved_lab_itemids': measurements['lab_itemids'],
    'resolved_chart_itemids': measurements['chart_itemids'],
}
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

In [ ]:
manifest_path = paths['manifests_dir'] / '03_cohort_construction_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='03_cohort_construction',
    config=config,
    extra={
        'cohort_summary': cohort_summary,
        'label_summary': label_summary,
        'saved_artifacts': saved_paths,
        'dictionary_tables_available': {
            'D_ITEMS.csv': 'D_ITEMS.csv' in available_tables,
            'D_LABITEMS.csv': 'D_LABITEMS.csv' in available_tables,
        },
    },
)
manifest_path

## Interpretation checklist

Before moving to feature engineering, review:

- How many ICU stays remain after the adult and ICU length filters?
- How often are suspected infection events identified?
- Which SOFA concepts resolved to item IDs successfully?
- Is the positive rate clinically plausible?
- Do any labeling assumptions need refinement before model training?

## Next notebook

`04_feature_engineering.ipynb` will transform this cohort into leakage-safe hourly structured trajectories for 6h, 12h, and 24h early prediction.